# Process APT experiment

## Label the dataset
For each experiment, there is a dedicated folder containing:
- .csv converted from .pcap file, containing extracted flows.
- .pkl contains time intervals in which attacks are performed (and additional info)
- .npz contains datetime objects that mark the start of the experiment.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import pickle

In [2]:
root = "./data/dollar_char_test/"
csv_file = root + "dollar_char_test.csv"
intervals = root + "dollar_char_test.pkl"
start = root + "dollar_char_test.npz"

### Load CSV network traffic

In [3]:
df = pd.read_csv(csv_file)

/tmp/ipykernel_2294825/298181477.py:1: DtypeWarning: Columns (0: requested_server_name, 1: client_fingerprint, 2: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


In [4]:
df.shape

(115219, 87)

In [5]:
list(df.columns)

['Unnamed: 0',
 'id',
 'expiration_id',
 'src_ip',
 'src_mac',
 'src_oui',
 'src_port',
 'dst_ip',
 'dst_mac',
 'dst_oui',
 'dst_port',
 'protocol',
 'ip_version',
 'vlan_id',
 'tunnel_id',
 'bidirectional_first_seen_ms',
 'bidirectional_last_seen_ms',
 'bidirectional_duration_ms',
 'bidirectional_packets',
 'bidirectional_bytes',
 'src2dst_first_seen_ms',
 'src2dst_last_seen_ms',
 'src2dst_duration_ms',
 'src2dst_packets',
 'src2dst_bytes',
 'dst2src_first_seen_ms',
 'dst2src_last_seen_ms',
 'dst2src_duration_ms',
 'dst2src_packets',
 'dst2src_bytes',
 'bidirectional_min_ps',
 'bidirectional_mean_ps',
 'bidirectional_stddev_ps',
 'bidirectional_max_ps',
 'src2dst_min_ps',
 'src2dst_mean_ps',
 'src2dst_stddev_ps',
 'src2dst_max_ps',
 'dst2src_min_ps',
 'dst2src_mean_ps',
 'dst2src_stddev_ps',
 'dst2src_max_ps',
 'bidirectional_min_piat_ms',
 'bidirectional_mean_piat_ms',
 'bidirectional_stddev_piat_ms',
 'bidirectional_max_piat_ms',
 'src2dst_min_piat_ms',
 'src2dst_mean_piat_ms',
 'sr

In [6]:
df['bidirectional_first_seen_ms']

0               17
1              218
2              298
3            21778
4            26386
            ...   
115214    35969562
115215    35976602
115216    35984640
115217    35990682
115218    35998720
Name: bidirectional_first_seen_ms, Length: 115219, dtype: int64

### Load experiment timestamps
We load the timestamps indicating the start and the end of each attacker phase/step.

In [7]:
import pickle
with open(intervals, 'rb') as f:
    loaded_data = pickle.load(f)

In [8]:
len(loaded_data)

310

In [9]:
loaded_data

[{'phase_number': 1,
  'step_number': 1,
  'attack_name': 'dollar_char_dos_scp_inst_cam_test',
  'phase_name': 'RECONNAISSANCE',
  'start_time': datetime.datetime(2025, 8, 20, 11, 38, 18, 722238, tzinfo=<UTC>),
  'end_time': datetime.datetime(2025, 8, 20, 11, 38, 18, 881681, tzinfo=<UTC>),
  'command': 'netstat',
  'iteration': 0},
 {'phase_number': -1,
  'step_number': 2,
  'attack_name': 'dollar_char_dos_scp_inst_cam_test',
  'phase_name': 'PAUSE',
  'start_time': datetime.datetime(2025, 8, 20, 11, 38, 18, 881726, tzinfo=<UTC>),
  'end_time': datetime.datetime(2025, 8, 20, 11, 40, 47, 329054, tzinfo=<UTC>),
  'command': 'pause',
  'iteration': 0},
 {'phase_number': 1,
  'step_number': 3,
  'attack_name': 'dollar_char_dos_scp_inst_cam_test',
  'phase_name': 'RECONNAISSANCE',
  'start_time': datetime.datetime(2025, 8, 20, 11, 40, 47, 329176, tzinfo=<UTC>),
  'end_time': datetime.datetime(2025, 8, 20, 11, 40, 47, 408082, tzinfo=<UTC>),
  'command': 'nmap_192_T4',
  'iteration': 0},
 {'p

### Count sequences for attack type

In [10]:
from collections import Counter
command_counts = Counter(item['command'] for item in loaded_data)
print(command_counts)

Counter({'pause': 164, 'dollar_char': 63, 'nmap_mqtt': 29, 'nmap_banner': 20, 'mqtt_cat': 9, 'scp_inst': 9, 'netstat': 5, 'nmap_192_T4': 5, 'nmap_10_T5': 5, 'brute_force_malformed': 1})


### Uniform time zone for start timestamp and experiment timestamps
Load start_time FROM .npy file. This is the starting timestamp of the apt simulation.

In [11]:
start_time = np.load(start, allow_pickle=True)
start_time = start_time['datetime']

In [12]:
start_time

array(datetime.datetime(2025, 8, 20, 13, 35, 53, 809608), dtype=object)

In [13]:
start_time = start_time.item().replace(hour=start_time.item().hour -2)

In [14]:
start_time

datetime.datetime(2025, 8, 20, 11, 35, 53, 809608)

### Convert relative timestamps to UTC

We convert the 'layers.frame.frame.time_epoch' relative time to absolute UTC timestamps using as reference the start_time we load from the .npy file.

In [15]:
from datetime import datetime, timedelta

def convert_to_absolute_time(epoch_seconds, start_time):
    utc=pytz.UTC

    relative_time = timedelta(milliseconds=epoch_seconds)
    
    absolute_time = start_time + relative_time
    absolute_time = utc.localize(absolute_time)
    return absolute_time

df['timestamp'] = df['bidirectional_first_seen_ms'].apply(convert_to_absolute_time, start_time=start_time)

In [16]:
df['timestamp'], df['bidirectional_first_seen_ms']

(0        2025-08-20 11:35:53.826608+00:00
 1        2025-08-20 11:35:54.027608+00:00
 2        2025-08-20 11:35:54.107608+00:00
 3        2025-08-20 11:36:15.587608+00:00
 4        2025-08-20 11:36:20.195608+00:00
                        ...               
 115214   2025-08-20 21:35:23.371608+00:00
 115215   2025-08-20 21:35:30.411608+00:00
 115216   2025-08-20 21:35:38.449608+00:00
 115217   2025-08-20 21:35:44.491608+00:00
 115218   2025-08-20 21:35:52.529608+00:00
 Name: timestamp, Length: 115219, dtype: datetime64[us, UTC],
 0               17
 1              218
 2              298
 3            21778
 4            26386
             ...   
 115214    35969562
 115215    35976602
 115216    35984640
 115217    35990682
 115218    35998720
 Name: bidirectional_first_seen_ms, Length: 115219, dtype: int64)

### Assign labels
- We assign labels based on phase name: if an observation begins between the start_time and the end_time of a step, it is marked with the corresponding attack name.
- Not labeled observations are marked as "discard": they correspond to sleep times between attack steps. 

In [17]:
import numpy as np
import pandas as pd

ph = pd.DataFrame(loaded_data)

print(ph) 

     phase_number  step_number                        attack_name  \
0               1            1  dollar_char_dos_scp_inst_cam_test   
1              -1            2  dollar_char_dos_scp_inst_cam_test   
2               1            3  dollar_char_dos_scp_inst_cam_test   
3              -1            4  dollar_char_dos_scp_inst_cam_test   
4               1            5  dollar_char_dos_scp_inst_cam_test   
..            ...          ...                                ...   
305            -1          306  dollar_char_dos_scp_inst_cam_test   
306             5          307  dollar_char_dos_scp_inst_cam_test   
307            -1          308  dollar_char_dos_scp_inst_cam_test   
308             5          309  dollar_char_dos_scp_inst_cam_test   
309            -1          310  dollar_char_dos_scp_inst_cam_test   

         phase_name                       start_time  \
0    RECONNAISSANCE 2025-08-20 11:38:18.722238+00:00   
1             PAUSE 2025-08-20 11:38:18.881726+00:00   
2  

In [18]:
ph[0:10]

,phase_number,step_number,attack_name,phase_name,start_time,end_time,command,iteration
0,1,1,dollar_char_dos_scp_inst_cam_test,RECONNAISSANCE,2025-08-20 11:38:18.722238+00:00,2025-08-20 11:38:18.881681+00:00,netstat,0
1,-1,2,dollar_char_dos_scp_inst_cam_test,PAUSE,2025-08-20 11:38:18.881726+00:00,2025-08-20 11:40:47.329054+00:00,pause,0
2,1,3,dollar_char_dos_scp_inst_cam_test,RECONNAISSANCE,2025-08-20 11:40:47.329176+00:00,2025-08-20 11:40:47.408082+00:00,nmap_192_T4,0
3,-1,4,dollar_char_dos_scp_inst_cam_test,PAUSE,2025-08-20 11:40:47.408113+00:00,2025-08-20 11:43:23.570797+00:00,pause,0
4,1,5,dollar_char_dos_scp_inst_cam_test,RECONNAISSANCE,2025-08-20 11:43:23.570862+00:00,2025-08-20 11:44:19.527836+00:00,nmap_10_T5,0
5,-1,6,dollar_char_dos_scp_inst_cam_test,PAUSE,2025-08-20 11:44:19.527872+00:00,2025-08-20 11:47:06.664863+00:00,pause,0
6,1,7,dollar_char_dos_scp_inst_cam_test,RECONNAISSANCE,2025-08-20 11:47:06.664902+00:00,2025-08-20 11:47:06.834350+00:00,netstat,0
7,-1,8,dollar_char_dos_scp_inst_cam_test,PAUSE,2025-08-20 11:47:06.834378+00:00,2025-08-20 11:49:35.232316+00:00,pause,0
8,1,9,dollar_char_dos_scp_inst_cam_test,RECONNAISSANCE,2025-08-20 11:49:35.232394+00:00,2025-08-20 11:49:35.318664+00:00,nmap_192_T4,0
9,-1,10,dollar_char_dos_scp_inst_cam_test,PAUSE,2025-08-20 11:49:35.318717+00:00,2025-08-20 11:51:26.277094+00:00,pause,0


In [19]:
# Ensure tz-aware UTC
ph['start_time'] = pd.to_datetime(ph['start_time'], utc=True)
ph['end_time']   = pd.to_datetime(ph['end_time'],   utc=True)
df['timestamp']  = pd.to_datetime(df['timestamp'],  utc=True)

In [20]:
df['timestamp']

0        2025-08-20 11:35:53.826608+00:00
1        2025-08-20 11:35:54.027608+00:00
2        2025-08-20 11:35:54.107608+00:00
3        2025-08-20 11:36:15.587608+00:00
4        2025-08-20 11:36:20.195608+00:00
                       ...               
115214   2025-08-20 21:35:23.371608+00:00
115215   2025-08-20 21:35:30.411608+00:00
115216   2025-08-20 21:35:38.449608+00:00
115217   2025-08-20 21:35:44.491608+00:00
115218   2025-08-20 21:35:52.529608+00:00
Name: timestamp, Length: 115219, dtype: datetime64[us, UTC]

In [21]:
ph['end_time']

0     2025-08-20 11:38:18.881681+00:00
1     2025-08-20 11:40:47.329054+00:00
2     2025-08-20 11:40:47.408082+00:00
3     2025-08-20 11:43:23.570797+00:00
4     2025-08-20 11:44:19.527836+00:00
                    ...               
305   2025-08-20 21:30:47.225071+00:00
306   2025-08-20 21:31:25.423780+00:00
307   2025-08-20 21:34:05.228898+00:00
308   2025-08-20 21:34:38.421490+00:00
309   2025-08-20 21:36:48.388408+00:00
Name: end_time, Length: 310, dtype: datetime64[us, UTC]

In [22]:
ph['start_time']

0     2025-08-20 11:38:18.722238+00:00
1     2025-08-20 11:38:18.881726+00:00
2     2025-08-20 11:40:47.329176+00:00
3     2025-08-20 11:40:47.408113+00:00
4     2025-08-20 11:43:23.570862+00:00
                    ...               
305   2025-08-20 21:28:00.130318+00:00
306   2025-08-20 21:30:47.225146+00:00
307   2025-08-20 21:31:25.423853+00:00
308   2025-08-20 21:34:05.228994+00:00
309   2025-08-20 21:34:38.421547+00:00
Name: start_time, Length: 310, dtype: datetime64[us, UTC]

In [23]:
iv = pd.IntervalIndex.from_arrays(ph['start_time'], ph['end_time'], closed='both')

In [24]:
iv

IntervalIndex([[2025-08-20 11:38:18.722238+00:00, 2025-08-20 11:38:18.881681+00:00],
               [2025-08-20 11:38:18.881726+00:00, 2025-08-20 11:40:47.329054+00:00],
               [2025-08-20 11:40:47.329176+00:00, 2025-08-20 11:40:47.408082+00:00],
               [2025-08-20 11:40:47.408113+00:00, 2025-08-20 11:43:23.570797+00:00],
               [2025-08-20 11:43:23.570862+00:00, 2025-08-20 11:44:19.527836+00:00],
               [2025-08-20 11:44:19.527872+00:00, 2025-08-20 11:47:06.664863+00:00],
               [2025-08-20 11:47:06.664902+00:00, 2025-08-20 11:47:06.834350+00:00],
               [2025-08-20 11:47:06.834378+00:00, 2025-08-20 11:49:35.232316+00:00],
               [2025-08-20 11:49:35.232394+00:00, 2025-08-20 11:49:35.318664+00:00],
               [2025-08-20 11:49:35.318717+00:00, 2025-08-20 11:51:26.277094+00:00],
               ...
               [2025-08-20 21:18:54.898211+00:00, 2025-08-20 21:21:49.775019+00:00],
               [2025-08-20 21:21:49.775076+00:

In [25]:
idx = iv.get_indexer(df['timestamp'])

In [26]:
idx

array([ -1,  -1,  -1, ..., 309, 309, 309], shape=(115219,))

In [27]:
df['phase_idx']     = idx
df['phase_name']    = np.where(idx >= 0, ph['phase_name'].to_numpy()[idx], None)
df['label']         = np.where(idx >= 0, ph['command'].to_numpy()[idx], 'discard')
df['phase_number']  = np.where(idx >= 0, ph['phase_number'].to_numpy()[idx], None)
df['step_number']   = np.where(idx >= 0, ph['step_number'].to_numpy()[idx], None)

In [28]:
df['label'].value_counts()

label
nmap_10_T5               105089
pause                      4797
nmap_mqtt                  2498
nmap_banner                1555
brute_force_malformed       596
dollar_char                 516
discard                     146
mqtt_cat                     11
scp_inst                     10
nmap_192_T4                   1
Name: count, dtype: int64

In [29]:
df['phase_name'].value_counts()

phase_name
RECONNAISSANCE    105090
PAUSE               4797
DISCOVERY           4064
BRUTE_FORCE          596
EXPLOIT              516
INSTALLATION          10
Name: count, dtype: int64

In [30]:
print(f"step number - {len(df['step_number'].value_counts().keys())}")
print("step_number")
for name, value in df['step_number'].value_counts().items():
    print(f"{name}:\t{value}")

step number - 300
step_number
5:	21036
11:	21023
23:	21012
17:	21009
29:	21009
31:	596
194:	95
192:	91
203:	91
61:	90
104:	90
53:	89
65:	89
256:	89
278:	89
170:	88
137:	87
181:	87
51:	86
213:	86
234:	86
59:	85
63:	85
67:	85
115:	85
159:	85
289:	85
57:	84
148:	84
223:	84
300:	84
2:	83
55:	83
93:	83
126:	83
245:	83
267:	82
106:	81
139:	80
247:	79
95:	78
172:	78
205:	78
225:	78
258:	77
269:	77
117:	76
128:	76
183:	76
280:	76
291:	76
161:	75
215:	75
236:	75
302:	75
150:	74
24:	73
101:	72
222:	72
251:	72
282:	71
163:	70
12:	68
64:	63
6:	62
40:	60
209:	55
38:	54
134:	49
211:	49
195:	45
204:	36
295:	36
147:	34
130:	33
227:	33
231:	33
52:	32
129:	32
187:	32
202:	32
233:	32
297:	32
301:	32
28:	31
88:	31
152:	31
169:	31
176:	31
220:	31
240:	31
4:	30
22:	30
50:	30
34:	30
62:	30
66:	30
154:	30
167:	30
182:	30
191:	30
193:	30
206:	30
244:	30
246:	30
249:	30
260:	30
270:	30
293:	30
14:	29
32:	29
136:	29
140:	29
160:	29
196:	29
198:	29
216:	29
257:	29
268:	29
281:	29
306:	29
54:	28
96:	28
103:	28
112

## Summary
Print a summary of the identified sequences for checking correctness.

In [31]:
import pandas as pd
import numpy as np

df = df.sort_values('timestamp').reset_index(drop=True)

step_summary = (
    df.groupby('step_number', as_index=False)
      .agg(
          label        = ('label', 'first'),
          phase_name   = ('phase_name', 'first'),
          phase_number = ('phase_number', 'first'),
          num_flws     = ('timestamp', 'size'),
          start_time   = ('timestamp', 'min'),
          end_time     = ('timestamp', 'max')
      )
      .assign(duration=lambda g: g['end_time'] - g['start_time'])
)

step_summary['fps'] = (
    step_summary['num_flws'] /
    step_summary['duration'].dt.total_seconds().replace(0, np.nan)
)

label_summary = (
    step_summary.groupby('label', as_index=False)
               .agg(
                   avg_num_flws    = ('num_flws', 'mean'),
                   median_num_flws = ('num_flws', 'median'),
                   avg_duration    = ('duration', 'mean'),
                   median_duration = ('duration', 'median'),
                   total_steps = ('step_number', 'nunique'),
                   total_flws      = ('num_flws', 'sum'),
                   avg_fps         = ('fps', 'mean')
               )
)


In [32]:
label_summary

,label,avg_num_flws,median_num_flws,avg_duration,median_duration,total_steps,total_flws,avg_fps
0,brute_force_malformed,596.000000,596.0,0 days 00:14:52.988000,0 days 00:14:52.988000,1,596,0.667422
1,dollar_char,8.190476,7.0,0 days 00:00:36.031682,0 days 00:00:26.966000,63,516,0.285905
2,mqtt_cat,1.375000,1.0,0 days 00:00:00.824125,0 days 00:00:00,8,11,1.119125
3,nmap_10_T5,21017.800000,21012.0,0 days 00:00:52.774000,0 days 00:00:53.478000,5,105089,399.136671
4,nmap_192_T4,1.000000,1.0,0 days 00:00:00,0 days 00:00:00,1,1,NaN
5,nmap_banner,77.750000,76.5,0 days 00:01:01.399050,0 days 00:01:01.758000,20,1555,1.268086
6,nmap_mqtt,86.137931,85.0,0 days 00:03:58.706000,0 days 00:03:58.915000,29,2498,0.360859
7,pause,29.250000,27.0,0 days 00:02:20.908237,0 days 00:02:21.189500,164,4797,0.206996
8,scp_inst,1.111111,1.0,0 days 00:00:00.004555,0 days 00:00:00,9,10,48.780488


In [33]:
step_summary

,step_number,label,phase_name,phase_number,num_flws,start_time,end_time,duration,fps
0,2,pause,PAUSE,-1,83,2025-08-20 11:38:23.472608+00:00,2025-08-20 11:40:41.336608+00:00,0 days 00:02:17.864000,0.602043
1,3,nmap_192_T4,RECONNAISSANCE,1,1,2025-08-20 11:40:47.375608+00:00,2025-08-20 11:40:47.375608+00:00,0 days 00:00:00,NaN
2,4,pause,PAUSE,-1,30,2025-08-20 11:40:55.427608+00:00,2025-08-20 11:43:21.458608+00:00,0 days 00:02:26.031000,0.205436
3,5,nmap_10_T5,RECONNAISSANCE,1,21036,2025-08-20 11:43:24.644608+00:00,2025-08-20 11:44:19.399608+00:00,0 days 00:00:54.755000,384.184093
4,6,pause,PAUSE,-1,62,2025-08-20 11:44:25.904608+00:00,2025-08-20 11:47:05.997608+00:00,0 days 00:02:40.093000,0.387275
...,...,...,...,...,...,...,...,...,...
295,306,pause,PAUSE,-1,29,2025-08-20 21:28:05.246608+00:00,2025-08-20 21:30:47.224608+00:00,0 days 00:02:41.978000,0.179037
296,307,dollar_char,EXPLOIT,5,7,2025-08-20 21:30:47.474608+00:00,2025-08-20 21:31:22.648608+00:00,0 days 00:00:35.174000,0.199011
297,308,pause,PAUSE,-1,28,2025-08-20 21:31:28.696608+00:00,2025-08-20 21:34:05.227608+00:00,0 days 00:02:36.531000,0.178878
298,309,dollar_char,EXPLOIT,5,6,2025-08-20 21:34:05.467608+00:00,2025-08-20 21:34:34.991608+00:00,0 days 00:00:29.524000,0.203224


## Remove Pauses 

In [34]:
df = df[df['label'] != 'pause']
df = df[df['label'] != 'discard']

In [35]:
df.shape

(110276, 93)

## Save labeled df to CSV

In [36]:
df.to_csv(root + 'dollar_char_test_labeled.csv', index=False)